# RSNA Knee Abnormality Detection — Phase 4C.1 synthetic second-reader canary

Internet-off Kaggle notebook. **Do not enable internet. GPU is required.**
This notebook scores **synthetic reports only**. It does **not** read the 58
gold studies or the 4,349 report-only studies. It does **not** train an image
model, create folds, or submit.

If any canary output is malformed, stop. Do not rerun the gold pilot.

Transport/parser protocol **v1.1**. Kernel version 7 remains the immutable
v1.0 `PILOT_EXECUTION_FAIL` run.


In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_second_reader_synthetic_canary"
MARKERS = ("knee/config.py", "knee/second_reader_pilot.py")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md", "literature/ontology.yaml"):
        src = src_root / filename
        if src.is_file():
            dest_file = dest / filename
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest_file)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


SKIP_WALK_DIRS = {"train_series", "test_series", "__pycache__", ".git", "competitions"}


def _walk_input_for_source(root: Path, max_depth: int = 6):
    zips: list[Path] = []
    dirs: list[Path] = []
    try:
        root_res = root.resolve()
    except OSError:
        return zips, dirs
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_res)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(
            d for d in dirnames if d not in SKIP_WALK_DIRS and d != COMPETITION_SLUG
        )
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            dirs.append(current)
        for name in filenames:
            if name.lower().endswith(".zip"):
                zip_path = current / name
                if _zip_has_source(zip_path):
                    zips.append(zip_path)
    return zips, dirs


def locate_and_extract_source() -> Path | None:
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips: list[Path] = []
    dirs: list[Path] = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found_zips, found_dirs = _walk_input_for_source(child)
            zips.extend(found_zips)
            dirs.extend(found_dirs)
    if zips:
        print(f"extracting_source_zip={zips[0]}")
        _safe_extract(zips[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        for nested in sorted(p for p in SRC_DST.iterdir() if p.is_dir()):
            if _looks_like_source(nested):
                return nested
    if dirs:
        print(f"copying_source_tree={dirs[0]}")
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
if SOURCE_ROOT is not None:
    resolved = str(SOURCE_ROOT.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"source_root={SOURCE_ROOT}")
else:
    print("source_root=not_found (will rely on an already importable knee package)")

try:
    from knee.kaggle_runtime import detect_kaggle_paths, KaggleLayout
    from knee.second_reader_canary import KAGGLE_CANARY_OUTPUT_DIRNAME
except ImportError as exc:
    raise SystemExit(
        "PILOT STOPPED: could not import knee. Attach the private source dataset. "
        f"Original error: {exc}"
    ) from exc

print("internet_used=False gpu_required=True training_run=False process_report_only=False synthetic_canary=True synthetic_canary=True")
print(detect_kaggle_paths(KaggleLayout(output_dirname=KAGGLE_CANARY_OUTPUT_DIRNAME)))


In [ ]:
# Synthetic canary. Do not read train.csv or print report text.
from pathlib import Path

from knee.report_labels import (
    HARD_WEAK_LABEL_TRAINING_ALLOWED,
    SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED,
)
from knee.second_reader_canary import run_synthetic_canary

assert HARD_WEAK_LABEL_TRAINING_ALLOWED is False
assert SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED is False

cfg = None
if SOURCE_ROOT is not None:
    cfg = str(Path(SOURCE_ROOT) / "configs" / "second_reader_pilot_v1_1.yaml")
WEIGHTS = os.environ.get("SECOND_READER_WEIGHTS")
OUT_DIR = Path("/kaggle/working/rsna_second_reader_synthetic_canary")

result = run_synthetic_canary(
    output_dir=OUT_DIR,
    config_path=cfg,
    weights_path=WEIGHTS,
)

print("status=", result.get("status"))
print("ok=", result.get("ok"))
print("n_examples=", result.get("n_examples"))
print("n_gold_studies_processed=", result.get("n_gold_studies_processed"))
print("n_report_only_studies_processed=", result.get("n_report_only_studies_processed"))
print("malformed_output_rate=", result.get("malformed_output_rate"))
print("all_parseable=", result.get("all_parseable"))
print("repeatable=", result.get("repeatable"))
print("scientific_evaluation=", result.get("scientific_evaluation"))
print("stop_gold_rerun=", result.get("stop_gold_rerun"))
print("inference_backend=", result.get("inference_backend"))
assert result.get("n_gold_studies_processed") == 0
assert result.get("n_report_only_studies_processed") == 0
assert result.get("scientific_evaluation") is False
if not result.get("ok"):
    raise SystemExit("CANARY_FAIL: do not rerun the 58-study gold pilot")


In [ ]:
# Display sanitized canary aggregates only.
from pathlib import Path
import json

out = Path(result["output_dir"])
sanitized = out / "sanitized"
print("sanitized_dir=", sanitized)
summary_path = sanitized / "second_reader_synthetic_canary_summary.md"
print(summary_path.read_text(encoding="utf-8")[:12000])
for name in ("freeze_manifest.json", "second_reader_synthetic_canary.json"):
    path = sanitized / name
    print(f"\n=== {name} ===")
    print(path.read_text(encoding="utf-8")[:4000])
